In [1]:
# =====================================================================
# [정리] WebRTC 평가를 별도 폴더로 분리
# 파인튜닝(학습)과 평가는 관심사가 다르므로 노트북·데이터를 나눈다
# =====================================================================
import shutil
from pathlib import Path

EVAL_DIR = Path.home()/'stt-finetune'/'eval-webrtc'
EVAL_DIR.mkdir(exist_ok=True)

for name in ['webrtc_eval', 'webrtc_raw']:
    src = Path.home()/'stt-finetune'/name
    if src.exists():
        shutil.move(str(src), str(EVAL_DIR/name))
        print(f"이동: {name}")

print(f"\n완료 -> {EVAL_DIR}")
print("이 폴더 안에 새 노트북을 만들어 아래 코드를 넣을 것")

이동: webrtc_eval
이동: webrtc_raw

완료 -> /home/j-i15a708/stt-finetune/eval-webrtc
이 폴더 안에 새 노트북을 만들어 아래 코드를 넣을 것


In [3]:
import os
from pathlib import Path
print("현재 위치:", os.getcwd())
print("\n[현재 폴더]")
for p in sorted(Path('.').iterdir()):
    print(" ", "📁" if p.is_dir() else "  ", p.name)
print("\n[webrtc_eval 검색]")
for p in Path.home().rglob('webrtc_eval'):
    print(" ", p)

현재 위치: /home/j-i15a708/stt-finetune/eval_webrtc

[현재 폴더]
  📁 .ipynb_checkpoints
     webrtc_eval.ipynb

[webrtc_eval 검색]
  /home/j-i15a708/stt-finetune/eval-webrtc/webrtc_eval


In [4]:
EV = Path.home()/'stt-finetune'/'eval-webrtc'/'webrtc_eval'
print("존재:", (EV/'manifest.jsonl').exists())

존재: True


In [5]:
import shutil
shutil.move(str(Path.home()/'stt-finetune'/'eval-webrtc'/'webrtc_eval'), './webrtc_eval')
shutil.rmtree(Path.home()/'stt-finetune'/'eval-webrtc', ignore_errors=True)
print("정리 완료")

정리 완료


In [6]:
ADAPTER = str(Path.home()/'stt-finetune'/'whisper-lora-exp5'/'final')
print("어댑터 존재:", Path(ADAPTER).exists())   # True 여야 함

어댑터 존재: True


In [7]:
# =====================================================================
# [D-12 검증] WebRTC 환경 A/B 평가 — 독립 실행 노트북
# ---------------------------------------------------------------------
# 학습 데이터(전화망 8kHz 협대역)로 파인튜닝한 모델이
# 실제 서비스 환경(브라우저 16kHz 광대역)에서도 효과를 유지하는지 확인.
# 같은 녹음을 baseline / 파인튜닝 모델에 각각 넣는 A/B 구조라
# 녹음 장비 차이는 양쪽에 동일하게 작용해 상쇄된다.
# =====================================================================
import os, json, torch, jiwer, soundfile as sf, librosa
from pathlib import Path
from collections import Counter, defaultdict
import transformers, warnings
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

BASE_MODEL  = "seastar105/whisper-medium-komixv2"
ADAPTER     = str(Path.home()/'stt-finetune'/'whisper-lora-exp5'/'final')  # 현재 최고 모델
EV          = Path('./webrtc_eval')

items = [json.loads(l) for l in
         (EV/'manifest.jsonl').read_text(encoding='utf-8').strip().split('\n') if l]
print(f"평가 샘플 {len(items)}건")
for spk, n in Counter(i['speaker'] for i in items).items():
    print(f"  {spk}: {n}건")
print("어댑터 존재:", Path(ADAPTER).exists())

평가 샘플 120건
  jaewon: 30건
  suhee: 30건
  junggyun: 30건
  junghyun: 30건
어댑터 존재: True


In [8]:
# =====================================================================
# [모델] 원본과 어댑터 적용본을 각각 준비
# ---------------------------------------------------------------------
# PeftModel은 원본 위에 어댑터를 얹는 구조이므로 원본을 두 번 로드한다.
# =====================================================================
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="korean", task="transcribe")

def load_base():
    m = WhisperForConditionalGeneration.from_pretrained(
        BASE_MODEL, torch_dtype=torch.float16).to("cuda")
    m.generation_config.language = "korean"
    m.generation_config.task = "transcribe"
    return m

base_model = load_base()                                  # 파인튜닝 전
ft_model   = PeftModel.from_pretrained(load_base(), ADAPTER)   # 파인튜닝 후
print("모델 로드 완료")

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

모델 로드 완료


In [11]:
# =====================================================================
# [평가] 전화망 평가셋과 동일 조건(beam=5)으로 측정해야 비교 가능
# =====================================================================
def load16k(p):
    y, sr = sf.read(p, dtype='float32')
    if y.ndim > 1: y = y.mean(axis=1)
    if sr != 16000: y = librosa.resample(y, orig_sr=sr, target_sr=16000)
    return y

def eval_webrtc(model, tag=""):
    model.eval(); preds, refs = [], []
    for n, it in enumerate(items):
        wav = load16k(EV/it['audio'])
        inp = processor(wav, sampling_rate=16000,
                        return_tensors="pt").input_features.to("cuda", torch.float16)
        with torch.no_grad():
            ids = model.generate(inp, max_new_tokens=200, num_beams=5)
        preds.append(processor.batch_decode(ids, skip_special_tokens=True)[0].strip())
        refs.append(it['text'].strip())
        if (n+1) % 30 == 0: print(f"  {tag} {n+1}/{len(items)}")
    return jiwer.cer(refs, preds), preds, refs

In [12]:
print("파인튜닝 전 측정...")
cer_base, preds_base, refs = eval_webrtc(base_model, "[base]")
print("\n파인튜닝 후 측정...")
cer_ft, preds_ft, _ = eval_webrtc(ft_model, "[ft]")

improve = (cer_base - cer_ft) / cer_base * 100
print("\n" + "="*58)
print(f"  WebRTC 평가셋 ({len(items)}건, 화자 4명)")
print(f"  파인튜닝 전 : {cer_base*100:.2f}%")
print(f"  파인튜닝 후 : {cer_ft*100:.2f}%")
print(f"  개선폭      : {improve:.1f}%")
print(f"  (참고) 전화망 평가셋 : 11.70% -> 6.10%, 47.9% 개선")
print("="*58)

for i in range(5):
    print(f"\n정답: {refs[i]}")
    print(f"전  : {preds_base[i]}")
    print(f"후  : {preds_ft[i]}")

파인튜닝 전 측정...
  [base] 30/120
  [base] 60/120
  [base] 90/120
  [base] 120/120

파인튜닝 후 측정...
  [ft] 30/120
  [ft] 60/120
  [ft] 90/120
  [ft] 120/120

  WebRTC 평가셋 (120건, 화자 4명)
  파인튜닝 전 : 9.89%
  파인튜닝 후 : 4.83%
  개선폭      : 51.1%
  (참고) 전화망 평가셋 : 11.70% -> 6.10%, 47.9% 개선

정답: 약관을 전달드리고 중요한 내용을 설명드리겠습니다.
전  : 약관을 전달드리고 중요한 내용을 설명드리겠습니다.
후  : 약관을 전달드리고 중요한 내용을 설명드리겠습니다.

정답: 이 약관 안에 보장되는 내용이랑 안 되는 내용이 다 적혀 있으니까 받으시면 한번 쭉 읽어보세요.
전  : 이 약관 안에 보장되는 내용이랑 안 되는 내용이 다 적혀 있으니까 받으시면 한번 쭉 읽어보세요.
후  : 이 약관 안에 보장되는 내용이랑 안 되는 내용이 다 적혀 있으니까 받으시면 한번 쭉 읽어보세요.

정답: 이 보장은 여든 살까지 받으실 수 있고요, 그 이후부터는 보장이 끝나요.
전  : 이 보장은 80살까지 받을 수 있고요 그 이후부터는 보장이 끝나요.
후  : 이 보장은 여든살까지 받을 수 있고요. 그 이후부터는 보장이 끝나요.

정답: 위험보장 기간은 첫 회 보험료 내신 날부터 시작된다고 보시면 됩니다.
전  : 위험보장기간은 첫해 보험료 내시 내신 날부터 시작된다고 보시면 됩니다.
후  : 위험보장기간은 첫 회 보험료 내신 날부터 시작된다고 보시면 됩니다.

정답: 중간에 해지하시면 그동안 낸 보험료보다 훨씬 적게 돌려받으실 수 있어요.
전  : 중간에 해지하시면 그동안 낸 보험료보다 훨씬 적게 돌려 받으실 수 있어요.
후  : 중간에 해지하시면 그동안 낸 보험료보다 훨씬 적게 돌려받으실 수 있어요.
